# End-to-End Sales Forecasting & Demand Intelligence System
**Advanced Sales Forecasting Project**

This notebook is built from scratch and covers all tasks sequentially:
1. **Task 1: Data Loading, Cleaning, Merging & Deep Exploration**
2. **Task 2: Time Series Analysis & Decomposition**
3. **Task 3: Sales Forecasting (SARIMA, Prophet, XGBoost) and Comparison**
4. **Task 4: Product Category & Region Level Forecasting**
5. **Task 5: Anomaly Detection (Isolation Forest & Z-Score)**
6. **Task 6: Product Demand Segmentation (Clustering & Stocking Strategies)**
7. **Task 7: Interactive Dashboard using Streamlit**
8. **Task 8: Executive Business Report**

## Import Libraries

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import xgboost as xgb

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
print("All libraries loaded successfully.")

---
## Task 1 — Data Loading, Cleaning, Merging & Deep Exploration

**Why this task?** Before building any forecasting model, we must first understand what data we have. Raw retail data is messy — it contains missing values, inconsistent dates, irrelevant columns, and potential outliers that can corrupt downstream models. Skipping this step is the #1 reason ML models fail in production. A business manager cannot trust a forecast built on uncleaned data.

**Business justification:** Every retail company (Walmart, Amazon, D-Mart) has a data engineering pipeline that cleans, validates, and enriches raw transactional data before it enters any analytics system. This task replicates that real-world pipeline — loading data from multiple sources, validating quality, cleaning issues, and performing exploratory analysis to understand sales patterns before any modeling begins.

### 1.1 Load Data
**Why?** We need to ingest the raw Superstore sales dataset (`train.csv`) and immediately inspect its structure — column names, data types, and shape — to understand what we're working with before any transformations.

In [ ]:
df = pd.read_csv('train.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
df.head()

### 1.2 Parse Dates & Extract Time Features
**Why?** The raw `Order Date` and `Ship Date` are stored as strings (object type). We must parse them into proper datetime objects so we can perform time-based aggregation (monthly/weekly grouping), calculate shipping durations, and extract temporal features (Year, Month, Quarter, Season) that are critical for seasonal analysis and as model inputs.

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')

df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Week'] = df['Order Date'].dt.isocalendar().week.astype(int)
df['Day of Week'] = df['Order Date'].dt.day_name()
df['Quarter'] = df['Order Date'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['Season'] = df['Month'].apply(get_season)
print("Time features extracted.")
df[['Order Date', 'Year', 'Month', 'Week', 'Day of Week', 'Quarter', 'Season']].head()

### 1.3 Data Quality Check
**Why?** Before cleaning, we must **diagnose** the problems. This step identifies missing values, duplicates, and provides a statistical summary (`describe()`) of the Sales column so we know the data's current state. This "before" snapshot lets us measure the impact of cleaning in the next step.

In [ ]:
print("=== DATA QUALITY REPORT ===")
print(f"\nDataset shape: {df.shape}")

print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

print(f"\nDate range: {df['Order Date'].min()} to {df['Order Date'].max()}")
print(f"Total Sales: ${df['Sales'].sum():,.2f}")
print(f"Unique Products: {df['Product Name'].nunique()}")
print(f"Unique Orders: {df['Order ID'].nunique()}")

print(f"\n=== Sales Distribution (Before Cleaning) ===")
print(df['Sales'].describe())

### 1.4 Data Cleaning

**Why clean before modeling?** The principle of "Garbage In, Garbage Out" applies directly here. If we feed a forecasting model data with missing postal codes, invalid shipping dates, or corrupt sales values, the model will learn noise instead of signal. Cleaning ensures that the patterns our models discover are **real business patterns**, not data artifacts.

**Cleaning steps (and why each matters):**
1. **Handle missing values** — 11 missing Postal Codes could cause errors in regional analysis. We fill using State-level mode (most common postal code in that state) rather than dropping rows, preserving data volume.
2. **Validate date consistency** — A Ship Date before an Order Date is physically impossible and indicates data entry errors. These rows must be removed as they corrupt shipping time calculations.
3. **Remove invalid Sales** — Zero or negative sales values are either returns, cancellations, or data errors. Including them would deflate monthly aggregates and mislead forecasting models.
4. **Outlier analysis (IQR)** — We detect extreme transactions to understand the sales distribution. However, we **keep** them because large B2B orders are legitimate business activity — removing them would understate actual revenue.
5. **Drop irrelevant columns** — `Row ID` is an arbitrary index; `Country` is constant (all "United States"). Keeping them adds noise and wastes memory.
6. **Sort and reset index** — Time series models require chronological ordering. Unsorted data would break lag-based features and decomposition.

In [ ]:
before_rows = len(df)
before_sales = df['Sales'].sum()
print(f"{'='*60}")
print(f"  DATA CLEANING PIPELINE")
print(f"{'='*60}")
print(f"Starting rows: {before_rows}")
print(f"Starting total sales: ${before_sales:,.2f}")

# ── Step 1: Handle Missing Postal Codes ──
missing_postal = df['Postal Code'].isnull().sum()
print(f"\n── Step 1: Missing Postal Codes ──")
print(f"   Found: {missing_postal} missing values")
if missing_postal > 0:
    # Show which States have missing postal codes
    print(f"   States affected: {df[df['Postal Code'].isnull()]['State'].unique()}")
    df['Postal Code'] = df.groupby('State')['Postal Code'].transform(
        lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 0)
    )
    print(f"   → Filled using State-level mode")
    print(f"   Remaining nulls: {df['Postal Code'].isnull().sum()}")

# ── Step 2: Validate Date Consistency ──
print(f"\n── Step 2: Date Validation ──")
invalid_dates = df[df['Ship Date'] < df['Order Date']]
print(f"   Rows with Ship Date < Order Date: {len(invalid_dates)}")
if len(invalid_dates) > 0:
    df = df[df['Ship Date'] >= df['Order Date']]
    print(f"   → Removed {len(invalid_dates)} invalid rows")
else:
    print(f"   → All dates valid ✓")

# ── Step 3: Remove Non-Positive Sales ──
print(f"\n── Step 3: Sales Value Validation ──")
non_positive = (df['Sales'] <= 0).sum()
print(f"   Rows with Sales <= 0: {non_positive}")
if non_positive > 0:
    df = df[df['Sales'] > 0]
    print(f"   → Removed {non_positive} rows")
else:
    print(f"   → All Sales values are positive ✓")

In [ ]:
# ── Step 4: Outlier Analysis (IQR Method) ──
print(f"── Step 4: Outlier Analysis (IQR Method) ──")
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['Sales'] < lower_bound) | (df['Sales'] > upper_bound)]

print(f"   Q1: ${Q1:.2f}")
print(f"   Q3: ${Q3:.2f}")
print(f"   IQR: ${IQR:.2f}")
print(f"   Lower bound: ${max(lower_bound, 0):.2f}")
print(f"   Upper bound: ${upper_bound:.2f}")
print(f"   Outliers detected: {len(outliers)} rows ({len(outliers)/len(df)*100:.1f}% of data)")

if len(outliers) > 0:
    print(f"\n   Top 5 outlier transactions:")
    print(outliers.nlargest(5, 'Sales')[['Order ID', 'Category', 'Sub-Category', 'Sales']].to_string(index=False))

print(f"\n   → Decision: KEEP outliers in the dataset")
print(f"     Reason: These are legitimate large B2B/corporate orders.")
print(f"     Removing them would understate actual revenue and produce")
print(f"     inaccurate monthly forecasts. Monthly aggregation naturally")
print(f"     smooths transaction-level variance.")

In [ ]:
# ── Step 5: Drop Irrelevant Columns ──
print(f"── Step 5: Drop Irrelevant Columns ──")
# Row ID: arbitrary index with no analytical value
# Country: all values are 'United States' (constant — zero variance)
cols_to_drop = ['Row ID', 'Country']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f"   Dropped: {cols_to_drop}")
print(f"   Remaining columns: {df.shape[1]}")

# ── Step 6: Sort by Date & Reset Index ──
print(f"\n── Step 6: Sort & Reset Index ──")
df = df.sort_values('Order Date').reset_index(drop=True)
print(f"   → Data sorted chronologically by Order Date")

# ══════════════════════════════════════════
#  CLEANING SUMMARY
# ══════════════════════════════════════════
after_rows = len(df)
after_sales = df['Sales'].sum()
print(f"\n{'='*60}")
print(f"  CLEANING SUMMARY")
print(f"{'='*60}")
print(f"  Rows before:  {before_rows}")
print(f"  Rows after:   {after_rows}")
print(f"  Rows removed: {before_rows - after_rows}")
print(f"  Columns:      {df.shape[1]}")
print(f"  Sales range:  ${df['Sales'].min():.2f} — ${df['Sales'].max():.2f}")
print(f"  Total Sales:  ${after_sales:,.2f}")
print(f"  Sales delta:  ${after_sales - before_sales:,.2f}")
print(f"{'='*60}")
print(f"  ✅ Data cleaning complete!")
print(f"{'='*60}")

In [ ]:
# Visualize Sales Distribution After Cleaning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box Plot
axes[0].boxplot(df['Sales'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#2196F3', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Sales Distribution (Box Plot)', fontsize=13)
axes[0].set_ylabel('Sales ($)')

# Histogram
axes[1].hist(df['Sales'], bins=50, color='#2196F3', edgecolor='black', alpha=0.7)
axes[1].axvline(x=df['Sales'].median(), color='red', linestyle='--', linewidth=2,
                label=f"Median: ${df['Sales'].median():.2f}")
axes[1].axvline(x=df['Sales'].mean(), color='orange', linestyle='--', linewidth=2,
                label=f"Mean: ${df['Sales'].mean():.2f}")
axes[1].set_title('Sales Distribution (Histogram)', fontsize=13)
axes[1].set_xlabel('Sales ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend(fontsize=11)

plt.suptitle('Sales Distribution After Cleaning', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('charts', exist_ok=True)
plt.savefig('charts/01b_sales_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Sales distribution plots saved to charts/01b_sales_distribution.png")

# Final cleaned data preview
print(f"\n=== Cleaned Data Preview ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

### 1.5 Aggregate into Weekly and Monthly Totals
**Why?** Our raw data has one row per line item (product in an order). For time series forecasting, we need data aggregated to regular intervals — **monthly** for the main models (48 data points over 4 years gives enough signal for SARIMA/Prophet) and **weekly** for higher-resolution trend analysis. This transforms transactional data into a proper time series.

In [ ]:
monthly_sales = df.groupby(pd.Grouper(key='Order Date', freq='MS')).agg(
    Sales=('Sales', 'sum'),
    Orders=('Order ID', 'nunique'),
    Avg_Sale=('Sales', 'mean')
)
monthly_sales = monthly_sales.asfreq('MS', fill_value=0)

weekly_sales = df.groupby(pd.Grouper(key='Order Date', freq='W')).agg(
    Sales=('Sales', 'sum'),
    Orders=('Order ID', 'nunique')
)

print(f"Monthly records: {len(monthly_sales)}")
print(f"Weekly records: {len(weekly_sales)}")
monthly_sales.info()
monthly_sales.head()

### 1.6 Exploratory Data Analysis (EDA) Questions
**Why?** EDA answers fundamental business questions that inform model design. For example: knowing that Technology leads revenue tells us where to focus segment forecasts; knowing Q4 spikes validates seasonal decomposition; knowing shipping takes ~4 days helps set fulfillment expectations. These are not just "interesting facts" — they directly shape downstream modeling decisions.

In [ ]:
# Q1: Which product category generates the most revenue?
cat_revenue = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print("=== Revenue by Category ===")
print(cat_revenue)

plt.figure(figsize=(8, 5))
cat_revenue.plot(kind='bar', color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='black')
plt.title('Total Revenue by Product Category', fontsize=14)
plt.ylabel('Sales ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('charts/01_category_revenue.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Q2: Which region has the most consistent sales growth over 4 years?
region_yearly = df.groupby(['Year', 'Region'])['Sales'].sum().unstack()
print("=== Yearly Sales by Region ===")
print(region_yearly)

# Calculate YoY growth consistency (lower std of growth = more consistent)
growth = region_yearly.pct_change().dropna()
growth_std = growth.std()
print(f"\nGrowth consistency (lower = more consistent):")
print(growth_std.sort_values())

plt.figure(figsize=(10, 6))
region_yearly.plot(kind='bar', ax=plt.gca())
plt.title('Yearly Sales by Region', fontsize=14)
plt.ylabel('Sales ($)')
plt.xticks(rotation=0)
plt.legend(title='Region')
plt.tight_layout()
plt.savefig('charts/02_region_yearly.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Q3: Average time between Order Date and Ship Date — and does it vary by region?
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days
avg_ship = df.groupby('Region')['Shipping Days'].mean()
print("=== Average Shipping Days by Region ===")
print(avg_ship.round(2))
print(f"\nOverall average: {df['Shipping Days'].mean():.2f} days")

In [ ]:
# Q4: Are there months that consistently spike across all years (seasonality)?
monthly_by_month = df.groupby(['Year', 'Month'])['Sales'].sum().unstack(level=0)
print("=== Monthly Sales Heatmap ===")

plt.figure(figsize=(12, 6))
sns.heatmap(monthly_by_month, annot=True, fmt=',.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Monthly Sales Heatmap by Year', fontsize=14)
plt.ylabel('Month')
plt.xlabel('Year')
plt.tight_layout()
plt.savefig('charts/03_seasonality_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()

# Average by month across all years
avg_monthly = df.groupby(['Year', 'Month'])['Sales'].sum().groupby('Month').mean()
spike_months = avg_monthly.nlargest(3)
print(f"\nTop 3 spike months (avg across years): {list(spike_months.index)}")

### 1.7 Supplementary Dataset — Video Game Sales (Multi-Source Analysis)
**Why?** In the real world, companies never analyze their sales data in isolation. They compare against industry benchmarks, competitor data, and macroeconomic trends. Here we load `vgsales.csv` (video game industry data) and merge with Superstore to demonstrate **multi-source data analysis** — a critical skill for business analysts who must correlate internal performance with external market conditions.

In [ ]:
vg = pd.read_csv('vgsales.csv')
vg.dropna(subset=['Year'], inplace=True)
vg['Year'] = vg['Year'].astype(int)
print(f"Video Game Sales dataset: {vg.shape}")
print(f"Columns: {list(vg.columns)}")
print(f"Year range: {vg['Year'].min()} - {vg['Year'].max()}")
vg.head()

In [ ]:
# Both datasets share a 'Year' column — merge on Year to compare industry trends
superstore_yearly = df.groupby('Year')['Sales'].sum().reset_index()
superstore_yearly.columns = ['Year', 'Superstore_Sales']
vg_yearly = vg.groupby('Year')['Global_Sales'].sum().reset_index()
vg_yearly.columns = ['Year', 'VG_Sales']
merged = pd.merge(superstore_yearly, vg_yearly, on='Year', how='inner')
print(f"Merged dataset ({len(merged)} years):")
print(merged)
corr = merged['Superstore_Sales'].corr(merged['VG_Sales'])
print(f"\nCorrelation between Superstore and Video Game sales: {corr:.3f}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.bar(merged['Year'] - 0.15, merged['Superstore_Sales'], width=0.3, color='#2196F3', label='Superstore')
ax1.set_ylabel('Superstore Sales ($)', color='#2196F3')
ax1.set_xlabel('Year')
ax2 = ax1.twinx()
ax2.bar(merged['Year'] + 0.15, merged['VG_Sales'], width=0.3, color='#FF9800', label='Video Games')
ax2.set_ylabel('Video Game Global Sales ($M)', color='#FF9800')
plt.title('Multi-Source Comparison: Retail vs Gaming Industry Sales', fontsize=14)
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.savefig('charts/03b_multi_source_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

**Observations from Multi-Source Analysis:**
- Both retail (Superstore) and gaming industry show overall growth over the years
- The correlation between the two industries helps validate sales trends
- This demonstrates real-world multi-source analysis: companies often combine internal data with industry benchmarks

---
## Task 2 — Time Series Analysis & Decomposition

**Why this task?** Before forecasting sales, we must understand the **underlying structure** of the time series. Is there a long-term trend? Is there seasonality? Is the series stationary? The answers to these questions directly determine which forecasting model to use and how to configure it.

**Business justification:** A store manager asking "Are sales going up or down?" needs the **trend**. A warehouse manager asking "Should I stock more in November?" needs the **seasonality**. A data scientist asking "Can I use ARIMA?" needs the **stationarity test**. Decomposition answers all three questions in one analysis — it separates the raw sales signal into Trend (long-term direction), Seasonal (repeating cycles), and Residual (unexplained noise), giving each stakeholder the specific insight they need.

**Technical justification:** The ADF stationarity test determines whether differencing is needed for ARIMA-family models. If the series has a unit root (non-stationary), we must difference it first. The seasonal period (12 months) informs the SARIMA seasonal order.

### 2.1 Plot Monthly Sales Trend
We plot the monthly sales aggregate from the **cleaned data** to observe the overall directional path of sales.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(monthly_sales.index, monthly_sales['Sales'], marker='o', linewidth=2, color='#2196F3')
plt.title('Monthly Sales Trend (2015-2018)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('charts/04_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.close()

### 2.2 Time Series Decomposition
We apply statsmodels `seasonal_decompose` using an additive model with a 12-month period to break the monthly sales trend into Trend, Seasonal, and Residual (noise) components.

In [ ]:
decomposition = seasonal_decompose(monthly_sales['Sales'], model='additive', period=12)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], title='Observed', color='#2196F3')
decomposition.trend.plot(ax=axes[1], title='Trend', color='#4CAF50')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal', color='#FF9800')
decomposition.resid.plot(ax=axes[3], title='Residual', color='#F44336')
plt.tight_layout()
plt.savefig('charts/05_decomposition.png', dpi=150, bbox_inches='tight')
plt.close()

### 2.3 Observations from Decomposition
- **Trend**: There is a clear upward long-term trend in sales from 2015 to 2018, validating business growth.
- **Seasonality**: Strong yearly seasonality is observed, with massive spikes in Q4 (specifically November and December) corresponding to holiday sales, and deep drops in Q1 (January/February).
- **Residual**: The residual noise is highest around November and December, suggesting that demand volatility is highest during peak shopping periods, which presents stock planning challenges.
- **Stationarity**: Despite the long-term trend, we need to statistically evaluate the presence of unit roots to assess stationarity.

### 2.4 Augmented Dickey-Fuller Test
The ADF test checks whether the monthly sales time series has a unit root (i.e., is non-stationary). A p-value < 0.05 means we reject the null hypothesis and confirm stationarity.

In [ ]:
ts = monthly_sales['Sales']
adf_result = adfuller(ts, autolag='AIC')
print("=== Augmented Dickey-Fuller Test ===")
print(f"Test Statistic: {adf_result[0]:.4f}")
print(f"p-value:        {adf_result[1]:.6f}")
print(f"Lags used:      {adf_result[2]}")
print(f"Observations:   {adf_result[3]}")
for key, val in adf_result[4].items():
    print(f"  Critical Value ({key}): {val:.4f}")

if adf_result[1] < 0.05:
    print(f"\n✅ Result: Series IS stationary (p={adf_result[1]:.4f} < 0.05)")
    print("   → No differencing required, but SARIMA's d=1 is still safe.")
else:
    print(f"\n⚠️ Result: Series is NOT stationary (p={adf_result[1]:.4f} >= 0.05)")
    print("   → Differencing required for ARIMA-based models.")

### 2.5 Task 2 Summary
- Monthly sales show a **clear upward trend** with **strong Q4 seasonality**
- The ADF test confirms the series behavior — this directly informs model selection
- The decomposition validates that both trend and seasonal components are significant, justifying the use of SARIMA (which handles both) and Prophet (which separately models trend + seasonality)

---
## Task 3 — Sales Forecasting using 3 Different Models

**Why this task?** This is the core of the project — predicting future sales so a business can make stocking, budgeting, and staffing decisions. We use **three different approaches** rather than just one because no single model is universally best. Comparing multiple models reveals which approach best captures this specific dataset's patterns.

**Why these 3 models specifically?**
1. **SARIMA** — The classical statistical approach. It explicitly models autoregressive patterns and seasonal cycles. Chosen because our decomposition (Task 2) confirmed strong 12-month seasonality. Serves as the **statistical baseline**.
2. **Facebook Prophet** — Developed by Meta for business forecasting at scale. It handles trend changepoints and multiple seasonalities automatically. Chosen because it's the **industry standard** for business time series and requires minimal tuning.
3. **XGBoost** — A machine learning approach that treats forecasting as a supervised regression problem using engineered lag features. Chosen because it can capture **non-linear patterns** and feature interactions that statistical models miss.

**Business justification:** In industry, you never deploy a model without comparing it against alternatives. A procurement team needs to know not just "what will sales be next month?" but "how confident are we in this number?" Comparing 3 models with MAE, RMSE, and MAPE metrics gives quantitative confidence in the chosen approach.

**Data leakage prevention:** For XGBoost, we use **recursive forecasting** — each prediction feeds back as input for the next step. This prevents the common mistake of using future actual values as features during testing, which would artificially inflate accuracy.

### 3.1 Train / Test Split
**Why?** We hold out the last 3 months as a test set to evaluate model accuracy on unseen data. This simulates the real-world scenario where a model is trained on historical data and must predict future months. Using only 3 months (instead of, say, 12) preserves maximum training data for the model while still providing a meaningful evaluation window.

In [ ]:
train = ts[:-3]
test = ts[-3:]
print(f"Training period: {train.index[0].strftime('%Y-%m')} to {train.index[-1].strftime('%Y-%m')} ({len(train)} months)")
print(f"Test period:     {test.index[0].strftime('%Y-%m')} to {test.index[-1].strftime('%Y-%m')} ({len(test)} months)")
print(f"\nTest actual values: {list(test.round(2))}")

### 3.2 Model 1 — SARIMA (1,1,1)(1,1,1)₁₂
**Why SARIMA?** Seasonal ARIMA captures both the autoregressive behavior (past values predict future values) and the 12-month seasonal cycle we confirmed in Task 2. The order (1,1,1) handles short-term patterns, while (1,1,1)₁₂ handles the yearly seasonality.

In [ ]:
sarima_model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
                       enforce_stationarity=False, enforce_invertibility=False)
sarima_results = sarima_model.fit(disp=False)
print(sarima_results.summary().tables[1])
sarima_forecast = sarima_results.get_forecast(steps=3)
sarima_pred = sarima_forecast.predicted_mean
sarima_ci = sarima_forecast.conf_int()
sarima_mae = mean_absolute_error(test, sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_pred))
sarima_mape = np.mean(np.abs((test - sarima_pred) / test)) * 100
print(f"\nSARIMA Metrics:")
print(f"  MAE:  ${sarima_mae:,.2f}")
print(f"  RMSE: ${sarima_rmse:,.2f}")
print(f"  MAPE: {sarima_mape:.2f}%")
print(f"  Forecast: {list(sarima_pred.round(2))}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(train.index, train, label='Training', color='#2196F3')
plt.plot(test.index, test, label='Actual (Test)', color='#4CAF50', marker='o')
plt.plot(sarima_pred.index, sarima_pred, label='SARIMA Forecast', color='#FF9800', marker='s', linestyle='--')
plt.fill_between(sarima_ci.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], alpha=0.2, color='#FF9800')
plt.title('SARIMA — Sales Forecast', fontsize=14)
plt.legend()
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/06_sarima_forecast.png', dpi=150, bbox_inches='tight')
plt.close()

### 3.3 Model 2 — Facebook Prophet
**Why Prophet?** Prophet fits a generalized additive model that automatically decomposes the series into Trend + Seasonality + Holidays. Unlike SARIMA, it handles trend changepoints (sudden shifts in growth rate) gracefully. It's the go-to model at companies like Meta, Uber, and Shopify for large-scale business forecasting.

In [ ]:
from prophet import Prophet
prophet_train = pd.DataFrame({
    'ds': train.index,
    'y': train.values
})
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
prophet_model.fit(prophet_train)
future = prophet_model.make_future_dataframe(periods=3, freq='MS')
prophet_forecast = prophet_model.predict(future)
prophet_pred = prophet_forecast[prophet_forecast['ds'].isin(test.index)]['yhat'].values
prophet_pred = pd.Series(prophet_pred, index=test.index)
prophet_mae = mean_absolute_error(test, prophet_pred)
prophet_rmse = np.sqrt(mean_squared_error(test, prophet_pred))
prophet_mape = np.mean(np.abs((test - prophet_pred) / test)) * 100
print(f"Prophet Metrics:")
print(f"  MAE:  ${prophet_mae:,.2f}")
print(f"  RMSE: ${prophet_rmse:,.2f}")
print(f"  MAPE: {prophet_mape:.2f}%")
print(f"  Forecast: {list(prophet_pred.round(2))}")

In [ ]:
fig = prophet_model.plot(prophet_forecast)
plt.title('Prophet — Sales Forecast', fontsize=14)
plt.tight_layout()
plt.savefig('charts/07_prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.close()

fig2 = prophet_model.plot_components(prophet_forecast)
plt.tight_layout()
plt.savefig('charts/07b_prophet_components.png', dpi=150, bbox_inches='tight')
plt.close()

### 3.4 Model 3 — XGBoost (ML-based with Recursive Forecasting)
**Why XGBoost?** Unlike SARIMA and Prophet (which are statistical models designed specifically for time series), XGBoost is a general-purpose machine learning model. We engineer lag features (Lags 1, 2, 3) and a rolling mean to convert the time series into a supervised regression task. XGBoost can then learn **non-linear relationships** between past and future sales that linear models cannot capture.

**Why recursive forecasting?** To prevent **data leakage**, we predict step-by-step: each prediction is appended to the history and used as the lag input for the next step. Without this, the model would "cheat" by using actual future values as input features during testing.

In [ ]:
def create_lag_features(data, lags=3):
    df_lag = pd.DataFrame({'Sales': data})
    for i in range(1, lags + 1):
        df_lag[f'Lag_{i}'] = df_lag['Sales'].shift(i)
    df_lag['Rolling_Mean_3'] = df_lag['Sales'].rolling(window=3).mean()
    df_lag['Month'] = df_lag.index.month
    df_lag['Quarter'] = df_lag.index.quarter
    df_lag['Season'] = df_lag['Month'].map({12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3})
    df_lag.dropna(inplace=True)
    return df_lag

train_lag_df = create_lag_features(train)
features = ['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3', 'Month', 'Quarter', 'Season']
target = 'Sales'
X_train_xgb = train_lag_df[features]
y_train_xgb = train_lag_df[target]

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_xgb, y_train_xgb)

# Recursive Forecasting (prevents data leakage)
history = list(train.values)
predictions = []
for i in range(3):
    lag_1 = history[-1]
    lag_2 = history[-2]
    lag_3 = history[-3]
    roll_mean = np.mean(history[-3:])
    pred_date = test.index[i]
    month = pred_date.month
    quarter = pred_date.quarter
    season = {12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3}[month]
    features_row = pd.DataFrame([[lag_1, lag_2, lag_3, roll_mean, month, quarter, season]], columns=features)
    pred = xgb_model.predict(features_row)[0]
    predictions.append(pred)
    history.append(pred)

xgb_pred = pd.Series(predictions, index=test.index)
xgb_mae = mean_absolute_error(test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(test, xgb_pred))
xgb_mape = np.mean(np.abs((test - xgb_pred) / test)) * 100
print(f"XGBoost Metrics:")
print(f"  MAE:  ${xgb_mae:,.2f}")
print(f"  RMSE: ${xgb_rmse:,.2f}")
print(f"  MAPE: {xgb_mape:.2f}%")
print(f"  Forecast: {list(xgb_pred.round(2))}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(train.index, train, label='Training', color='#2196F3')
plt.plot(test.index, test, label='Actual (Test)', color='#4CAF50', marker='o')
plt.plot(xgb_pred.index, xgb_pred, label='XGBoost Forecast', color='#9C27B0', marker='^', linestyle='--')
plt.title('XGBoost — Sales Forecast', fontsize=14)
plt.legend()
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/08_xgboost_forecast.png', dpi=150, bbox_inches='tight')
plt.close()

### 3.5 Model Comparison
**Why compare?** We evaluate all 3 models using 3 standard metrics — **MAE** (average dollar error), **RMSE** (penalizes large errors), and **MAPE** (percentage error for business-friendly interpretation). The model with the lowest MAPE is recommended for production because MAPE is scale-independent and most interpretable by business stakeholders ("the model is off by X%").

In [ ]:
comparison = pd.DataFrame({
    'Model': ['SARIMA', 'Prophet', 'XGBoost'],
    'MAE': [sarima_mae, prophet_mae, xgb_mae],
    'RMSE': [sarima_rmse, prophet_rmse, xgb_rmse],
    'MAPE (%)': [sarima_mape, prophet_mape, xgb_mape],
    'Month 1': [sarima_pred.iloc[0], prophet_pred.iloc[0], xgb_pred.iloc[0]],
    'Month 2': [sarima_pred.iloc[1], prophet_pred.iloc[1], xgb_pred.iloc[1]],
    'Month 3': [sarima_pred.iloc[2], prophet_pred.iloc[2], xgb_pred.iloc[2]]
  })
print("=== MODEL COMPARISON TABLE ===")
print(comparison.to_string(index=False))
best_model = comparison.loc[comparison['MAPE (%)'].idxmin(), 'Model']
best_mape = comparison['MAPE (%)'].min()
print(f"\nBest Model: {best_model} (lowest MAPE: {best_mape:.2f}%)")

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(ts.index, ts, label='Actual', color='black', linewidth=2)
plt.plot(sarima_pred.index, sarima_pred, label=f'SARIMA (MAPE={sarima_mape:.1f}%)', marker='s', linestyle='--', color='#FF9800')
plt.plot(prophet_pred.index, prophet_pred, label=f'Prophet (MAPE={prophet_mape:.1f}%)', marker='o', linestyle='--', color='#2196F3')
plt.plot(xgb_pred.index, xgb_pred, label=f'XGBoost (MAPE={xgb_mape:.1f}%)', marker='^', linestyle='--', color='#9C27B0')
plt.axvline(x=test.index[0], color='gray', linestyle=':', alpha=0.5, label='Train/Test Split')
plt.title('3-Month Forecast Comparison — All Models', fontsize=14)
plt.legend(fontsize=11)
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('charts/09_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

### 3.6 Production Model Recommendation
Based on the numerical evaluation above, we recommend the model with the **lowest MAPE** for production forecasting:
- The best model achieves the lowest errors across MAE, RMSE, and MAPE on the held-out test set.
- XGBoost's tree-based nature allows it to learn non-linear patterns, interaction effects, and seasonal structures more robustly than linear statistical models.
- With the recursive forecasting framework, it prevents data leakage and is highly customizable for adding new features (e.g. promotional flags, inventory levels).
- SARIMA and Prophet serve as strong baselines for comparison and provide interpretable trend/seasonality decompositions.

---
## Task 4 — Product Category & Region Level Forecasting

**Why this task?** A single aggregate forecast tells you total company sales, but it does NOT tell you **what to stock where**. A warehouse in the West Region needs West Region forecasts. The Furniture buyer needs Furniture-specific forecasts. Segment-level forecasting converts a company-wide number into **actionable, department-level decisions**.

**Business justification:** Consider this real-world scenario — the aggregate forecast says "sales will be $80K next month." But is that $40K Furniture + $20K Technology + $20K Office Supplies? Or $10K + $50K + $20K? The stocking decisions are completely different. A store that over-stocks Furniture and under-stocks Technology loses sales on high-demand items while capital sits tied up in slow-moving inventory.

**Why 5 segments?** We forecast across two dimensions that matter most for operations:
- **3 Product Categories** (Furniture, Technology, Office Supplies) — drives procurement and inventory allocation
- **2 Key Regions** (West, East) — drives warehouse distribution and regional staffing

**Technical approach:** We reuse the best model from Task 3 (XGBoost with recursive forecasting) and train it independently on each segment's monthly time series. Each segment has its own demand patterns, seasonality, and growth trajectory.

In [ ]:
segments = {
    'Furniture': df[df['Category'] == 'Furniture'],
    'Technology': df[df['Category'] == 'Technology'],
    'Office Supplies': df[df['Category'] == 'Office Supplies'],
    'West Region': df[df['Region'] == 'West'],
    'East Region': df[df['Region'] == 'East']
}

segment_forecasts = {}
segment_monthly = {}
features = ['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3', 'Month', 'Quarter', 'Season']

for name, seg_df in segments.items():
    monthly = seg_df.groupby(pd.Grouper(key='Order Date', freq='MS')).agg(Sales=('Sales', 'sum'))
    monthly = monthly.asfreq('MS', fill_value=0)
    segment_monthly[name] = monthly['Sales']
    
    ts_seg = monthly['Sales']
    train_seg = ts_seg[:-3]
    test_seg = ts_seg[-3:]
    
    train_lag_seg = create_lag_features(train_seg)
    X_train_seg = train_lag_seg[features]
    y_train_seg = train_lag_seg['Sales']
    
    xgb_seg = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
    xgb_seg.fit(X_train_seg, y_train_seg)
    
    history = list(train_seg.values)
    predictions = []
    for i in range(3):
        lag_1 = history[-1]
        lag_2 = history[-2]
        lag_3 = history[-3]
        roll_mean = np.mean(history[-3:])
        pred_date = test_seg.index[i]
        month = pred_date.month
        quarter = pred_date.quarter
        season = {12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3}[month]
        features_row = pd.DataFrame([[lag_1, lag_2, lag_3, roll_mean, month, quarter, season]], columns=features)
        pred = xgb_seg.predict(features_row)[0]
        predictions.append(pred)
        history.append(pred)
        
    segment_forecasts[name] = pd.Series(predictions, index=test_seg.index)
    print(f"{name} Segment Model trained and forecasted.")

growth_rates = {}
for name, forecast in segment_forecasts.items():
    future_forecast = forecast.mean()
    past_avg = segment_monthly[name].tail(12).mean()
    growth = ((future_forecast - past_avg) / past_avg) * 100
    growth_rates[name] = growth
    print(f"{name}: Forecasted avg = ${future_forecast:,.2f}, Past 12mo avg = ${past_avg:,.2f}, Growth = {growth:.1f}%")

best_growth = max(growth_rates, key=growth_rates.get)
print(f"\nStrongest upcoming growth according to model: {best_growth} ({growth_rates[best_growth]:.1f}%)")

In [ ]:
plt.figure(figsize=(14, 7))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
for idx, (name, forecast) in enumerate(segment_forecasts.items()):
    plt.plot(forecast.index, forecast.values, color=colors[idx], marker='o', linestyle='--', label=f'{name} Forecast')
plt.title('3-Month Segment Forecasts Comparison (XGBoost)', fontsize=14)
plt.ylabel('Sales ($)')
plt.legend()
plt.tight_layout()
plt.savefig('charts/10_segment_forecasts.png', dpi=150, bbox_inches='tight')
plt.close()

---
## Task 5 — Anomaly Detection in Sales Data

**Why this task?** Anomaly detection helps identify unusual sales spikes or drops that deviate from normal patterns. These anomalies can represent critical business events: operational errors (e.g., pricing mistakes), bulk B2B purchases, or seasonal promotions. Detecting and understanding anomalies allows business managers to separate noise/one-off events from predictable trends.

**Business justification:** If a retail store sees a sudden $50,000 spike in weekly sales, is it the "new normal," a seasonal trend, or a one-time anomaly? If it's a one-time anomaly (like a corporate bulk order), the warehouse should *not* restock based on that spike, or they will end up with dead inventory. Anomaly detection guides stocking decisions by flagging which historical data points are outliers.

**Why two different methods?** We compare two complementary techniques:
1. **Isolation Forest** — An unsupervised machine learning algorithm that isolates anomalies by randomly partitioning features. It is robust and captures multivariate or complex, non-linear outliers.
2. **Z-Score** — A statistical method measuring how many standard deviations a point is from the rolling mean. It is simple, highly interpretable, and perfect for identifying extreme univariate fluctuations relative to a local window.

### 5.1 Isolation Forest
**Why Isolation Forest?** Isolation Forest is a tree-based model that works on the principle that anomalies are few and different, making them easier to "isolate" (require fewer splits in a random tree). We apply it to our weekly sales data with a contamination rate of 5% to flag the most extreme 5% of weeks.

In [ ]:
weekly_data = weekly_sales[['Sales']].copy()
weekly_data['Week_Index'] = range(len(weekly_data))
iso_forest = IsolationForest(contamination=0.05, random_state=42)
weekly_data['Anomaly_IF'] = iso_forest.fit_predict(weekly_data[['Sales']])
weekly_data['Anomaly_IF'] = weekly_data['Anomaly_IF'].map({1: 0, -1: 1})
anomalies_if = weekly_data[weekly_data['Anomaly_IF'] == 1]
print(f"Isolation Forest detected {len(anomalies_if)} anomalous weeks out of {len(weekly_data)}")
print(f"\nAnomaly weeks:")
print(anomalies_if[['Sales']])

In [ ]:
for date, row in anomalies_if.iterrows():
    month = date.month
    year = date.year
    if month in [11, 12]:
        reason = f"Sales spike in {date.strftime('%B %Y')} likely corresponds to festive/holiday sale period (Black Friday, Christmas peaks)"
    elif month in [1, 2]:
        reason = f"Unusually low sales in {date.strftime('%B %Y')} — post-holiday demand slump (low consumer spending)"
    elif month in [9, 10]:
        reason = f"Elevated sales in {date.strftime('%B %Y')} — back-to-school or early holiday prep order spikes"
    else:
        reason = f"Unusual pattern in {date.strftime('%B %Y')} — possible corporate bulk order or mid-year promotion"
    print(f"  {date.strftime('%Y-%m-%d')}: Sales=${row['Sales']:,.2f} — {reason}")

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(weekly_data.index, weekly_data['Sales'], color='#2196F3', linewidth=1, label='Weekly Sales')
plt.scatter(anomalies_if.index, anomalies_if['Sales'], color='#F44336', s=100, zorder=5, label='Isolation Forest Anomaly', marker='x')
plt.title('Isolation Forest — Anomaly Detection', fontsize=14)
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.savefig('charts/11_anomaly_isolation_forest.png', dpi=150, bbox_inches='tight')
plt.close()

### 5.2 Z-Score Based Detection
**Why Z-Score?** A simple global Z-Score is sensitive to trend and seasonality. Instead, we use a **12-week rolling window Z-Score**. This local Z-Score measures deviation relative to adjacent weeks, effectively adjusting for the overall upward trend and holiday seasonality. We flag anomalies where the absolute Z-Score is greater than 2.

In [ ]:
window = 12  # 12-week rolling window
weekly_data['Rolling_Mean'] = weekly_data['Sales'].rolling(window=window, center=True).mean()
weekly_data['Rolling_Std'] = weekly_data['Sales'].rolling(window=window, center=True).std()
weekly_data['Z_Score'] = (weekly_data['Sales'] - weekly_data['Rolling_Mean']) / weekly_data['Rolling_Std'].replace(0, np.nan).fillna(1)

weekly_data['Anomaly_ZS'] = (weekly_data['Z_Score'].abs() > 2).astype(int)
anomalies_zs = weekly_data[weekly_data['Anomaly_ZS'] == 1]
print(f"Z-Score detected {len(anomalies_zs)} anomalous weeks")
print(anomalies_zs[['Sales', 'Z_Score']])

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(weekly_data.index, weekly_data['Sales'], color='#2196F3', linewidth=1, label='Weekly Sales')
plt.scatter(anomalies_zs.index, anomalies_zs['Sales'], color='#FF9800', s=100, zorder=5, label='Z-Score Anomaly', marker='D')
plt.title('Z-Score — Anomaly Detection', fontsize=14)
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.savefig('charts/12_anomaly_zscore.png', dpi=150, bbox_inches='tight')
plt.close()

### 5.3 Comparison
**Why compare?** Comparing the two methods helps us understand their strengths. Isolation Forest takes a global/multivariate perspective (isolating points across the whole distribution), whereas our rolling Z-Score evaluates points relative to their local temporal context. Understanding which anomalies are flagged by both or only one method allows us to build a more robust detection pipeline.

In [ ]:
if_only = set(anomalies_if.index) - set(anomalies_zs.index)
zs_only = set(anomalies_zs.index) - set(anomalies_if.index)
both = set(anomalies_if.index) & set(anomalies_zs.index)

print(f"Anomalies flagged by BOTH methods: {len(both)} weeks")
print(f"  Dates: {[d.strftime('%Y-%m-%d') for d in sorted(both)]}")
print(f"\nIsolation Forest only: {len(if_only)} weeks")
print(f"  Dates: {[d.strftime('%Y-%m-%d') for d in sorted(if_only)]}")
print(f"\nZ-Score only: {len(zs_only)} weeks")
print(f"  Dates: {[d.strftime('%Y-%m-%d') for d in sorted(zs_only)]}")
print(f"\nInterpretation: Methods partially agree, but Isolation Forest captures multivariate patterns")
print(f"while Z-Score focuses on statistical outliers. Both agree on extreme anomalies.")

### 5.4 Multi-Source Anomaly Detection — Video Game Sales
**Why this analysis?** To demonstrate the robustness of our anomaly detection, we apply Isolation Forest to the supplementary video game sales dataset (`vgsales.csv`). We aggregate sales by year, detect anomalous sales years, and compare whether Superstore anomaly years align with video game sales anomaly years. This helps check for broader macroeconomic/industry-wide patterns.

In [ ]:
vg_yearly_sales = vg.groupby('Year')['Global_Sales'].sum().reset_index()
vg_yearly_sales.set_index('Year', inplace=True)
iso_vg = IsolationForest(contamination=0.1, random_state=42)
vg_yearly_sales['Anomaly'] = iso_vg.fit_predict(vg_yearly_sales[['Global_Sales']])
vg_yearly_sales['Anomaly'] = vg_yearly_sales['Anomaly'].map({1: 0, -1: 1})
anomalies_vg = vg_yearly_sales[vg_yearly_sales['Anomaly'] == 1]
print(f"Video Game Sales: {len(anomalies_vg)} anomalous years detected")
print(anomalies_vg)

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(vg_yearly_sales.index, vg_yearly_sales['Global_Sales'], color='#2196F3', alpha=0.7, label='Normal')
plt.bar(anomalies_vg.index, anomalies_vg['Global_Sales'], color='#F44336', label='Anomaly')
plt.title('Video Game Sales — Anomaly Detection (Isolation Forest)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Global Sales ($M)')
plt.legend()
plt.tight_layout()
plt.savefig('charts/12b_vg_anomaly.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
superstore_anomaly_years = set(d.year for d in anomalies_if.index)
vg_anomaly_years = set(anomalies_vg.index)
both_anomaly = superstore_anomaly_years & vg_anomaly_years
print(f"Superstore anomaly years: {sorted(superstore_anomaly_years)}")
print(f"Video Game anomaly years: {sorted(vg_anomaly_years)}")
print(f"Years with anomalies in BOTH industries: {sorted(both_anomaly) if both_anomaly else 'None'}")
if both_anomaly:
    print("This suggests external economic factors affecting both retail and gaming sectors.")

---
## Task 6 — Product Demand Segmentation using Clustering

**Why this task?** A retail store has thousands of products with completely different behaviors. Some are high-revenue and stable (e.g., Copiers, Chairs), while others are low-volume but highly volatile (e.g., Fasteners, Envelopes). Standardizing stocking strategies across all products leads to stockouts or bloated storage. Clustering groups products with similar demand profiles, allowing us to define customized stocking strategies for each segment.

**Business justification:** Managing inventory for 1,800+ products individually is impossible. But grouping them into 3 distinct "Demand Segments" (e.g., Stable Giants, Volatile Starters, Growing Stars) simplifies operations. Warehouse managers can assign safety stock multipliers and review periods based on the cluster label, maximizing capital efficiency.

**Technical approach:**
1. **Feature Engineering**: Compute Total Sales, YoY Growth Rate, Volatility (std), and Average Order Value for all 17 sub-categories.
2. **Standardization**: Scale features since KMeans uses Euclidean distance (scales differ between sales dollars and growth rates).
3. **Elbow Method**: Use inertia to select K=3 as the optimal number of segments.
4. **K-Means & PCA**: Fit KMeans and project features onto a 2D space using PCA for clear business visualization.

### 6.1 Feature Engineering at Sub-Category Level
**Why?** We group sales chronologically by month for each product sub-category. To represent their demand profile, we extract: Total Sales, YoY growth rate, demand Volatility, and Average Order Value. Sub-categories with less than 12 months of sales are excluded to avoid sample size bias.

In [ ]:
subcat = df.groupby(['Sub-Category', pd.Grouper(key='Order Date', freq='MS')])['Sales'].sum().reset_index()
subcat_features = []
for name, group in subcat.groupby('Sub-Category'):
    group = group.sort_values('Order Date')
    monthly_sales_s = group.set_index('Order Date')['Sales']
    if len(monthly_sales_s) < 12:
        continue
    total_sales = monthly_sales_s.sum()
    avg_order_value = monthly_sales_s.mean()
    yearly = monthly_sales_s.resample('YS').sum()
    if len(yearly) > 1:
        growth_rate = (yearly.iloc[-1] - yearly.iloc[0]) / yearly.iloc[0] * 100
    else:
        growth_rate = 0
    volatility = monthly_sales_s.std()
    subcat_features.append({
        'Sub-Category': name,
        'Total_Sales': total_sales,
        'Growth_Rate': growth_rate,
        'Volatility': volatility,
        'Avg_Order_Value': avg_order_value
    })
features_df = pd.DataFrame(subcat_features)
print(f"Features computed for {len(features_df)} sub-categories")
features_df

### 6.2 Elbow Method to Find Optimal Clusters
**Why?** The Elbow Method evaluates the sum of squared distances of samples to their closest cluster center (inertia) for a range of K. We look for the "elbow" point where adding more clusters provides diminishing returns in explaining variance. This prevents over-clustering.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_df[['Total_Sales', 'Growth_Rate', 'Volatility', 'Avg_Order_Value']])
inertias = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Optimal K')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('charts/13_elbow_method.png', dpi=150, bbox_inches='tight')
plt.close()
print("Choose K where the elbow bends — typically K=3 or K=4")

### 6.3 K-Means Clustering & PCA Visualization
**Why?** We segment into K=3 clusters based on the elbow plot. Since our feature space has 4 dimensions, we apply **Principal Component Analysis (PCA)** to project the clusters onto a 2D plane. This lets us visually audit our clusters, ensure they are well-separated, and annotate product groups in a way business leaders can easily grasp.

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
features_df['Cluster'] = kmeans.fit_predict(X_scaled)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
features_df['PC1'] = X_pca[:, 0]
features_df['PC2'] = X_pca[:, 1]
cluster_profiles = features_df.groupby('Cluster')[['Total_Sales', 'Growth_Rate', 'Volatility']].mean()
print("Cluster Profiles:")
print(cluster_profiles)

# Map labels dynamically to prevent duplicate mapping and ensure clean categories
cluster_labels = {}
available_clusters = list(range(optimal_k))
high_vol_c = cluster_profiles['Total_Sales'].idxmax()
cluster_labels[high_vol_c] = 'High Volume, Stable Demand'
available_clusters.remove(high_vol_c)

remaining_growth = cluster_profiles.loc[available_clusters, 'Growth_Rate']
growing_c = remaining_growth.idxmax()
cluster_labels[growing_c] = 'Growing Demand'
available_clusters.remove(growing_c)

for c in available_clusters:
    cluster_labels[c] = 'Low Volume, High Volatility'

features_df['Cluster_Label'] = features_df['Cluster'].map(cluster_labels)
print(f"\nCluster Labels: {cluster_labels}")

In [ ]:
plt.figure(figsize=(10, 7))
colors_cluster = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for c in range(optimal_k):
    mask = features_df['Cluster'] == c
    plt.scatter(features_df[mask]['PC1'], features_df[mask]['PC2'],
                s=150, c=colors_cluster[c], label=cluster_labels[c], alpha=0.8, edgecolors='white')
    for _, row in features_df[mask].iterrows():
        plt.annotate(row['Sub-Category'], (row['PC1'], row['PC2']),
                     fontsize=8, ha='left', va='bottom')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Product Demand Segments — PCA Visualization', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('charts/14_clusters_pca.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
for c in range(optimal_k):
    items = features_df[features_df['Cluster'] == c]['Sub-Category'].tolist()
    print(f"\n{cluster_labels[c]}:")
    for item in items:
        row = features_df[features_df['Sub-Category'] == item].iloc[0]
        print(f"  - {item}: Total=${row['Total_Sales']:,.0f}, Growth={row['Growth_Rate']:.1f}%, Vol=${row['Volatility']:,.0f}")

### 6.4 Stocking Strategy Recommendations
Based on the clustered demand segments, we prescribe customized stocking strategies to optimize capital and shelf space:

- **High Volume, Stable Demand (Safety Stock Focus)**: These products (e.g. Chairs, Binders, Phones) generate bulk revenue. Maintain higher safety stock levels (~2 weeks of demand) to prevent stockouts and use automated reorder points.
- **Low Volume, High Volatility (JIT / Alert Focus)**: These products (e.g. Art, Fasteners, Supplies) have erratic demand. Keep safety stock minimal (~3 days of demand) or employ a **Just-In-Time (JIT)** model to avoid tying up capital in slow-moving inventory.
- **Growing Demand (Growth Buffer Focus)**: For products with explosive sales growth (e.g. Copiers), hold safety stock slightly higher than current averages (~1.5 weeks) to capture incoming growth and review replenishment levels monthly.

---
## Task 7 — Interactive Dashboard using Streamlit

**Why this task?** While static charts and tables in a notebook are useful for analysis, business managers need an interactive tool to explore demand projections, filter regional performance, and inspect anomaly reports dynamically on a weekly/monthly basis. Delivering everything through a dashboard enables non-technical decision makers to utilize our forecasting models without reading code.

**Dashboard implementation details:** We created a premium, production-ready Streamlit application in [app.py](file:///c:/Users/Admin/Desktop/xylofi/week-3/SalesForecasting_Superstore/app.py) featuring a dark glassmorphism aesthetic and curated interactive Plotly components. It includes four primary views:
1. **Sales Overview**: A high-level visual summary of revenue, orders, customer acquisition, and shipment duration metrics with interactive hierarchical sunburst breakdowns.
2. **Forecast Explorer**: Interactive selector to generate Prophet machine learning sales forecasts for specific regions and product categories over a variable 1-6 month horizon, including validation metrics (MAE, RMSE, MAPE) computed via historical backtesting.
3. **Anomaly Report**: Unsupervised anomaly detection report comparing Isolation Forest flags against rolling Z-Score statistical thresholds, complete with hover tooltips and dynamic contextual explanation mapping.
4. **Product Demand Segments**: Visual cluster exploration (PCA scatter plot) separating product categories into risk-adjusted segments with integrated procurement guidelines.

**How to run the dashboard:**
To start the dashboard locally, execute the following command in your terminal:
```bash
streamlit run app.py
```

---
## Task 8 — Executive Business Report

**Why this task?** A project's technical achievements must be compiled into a high-level summary that translates model metrics and statistical outliers into business value. Supply Chain Heads and CFOs require synthesized takeaways, expected forecasts, anomalous week warnings, and inventory strategies to make final strategic decisions.

**Business report overview:** The complete executive report has been compiled and rendered as a publication-quality PDF at [summary.pdf](file:///c:/Users/Admin/Desktop/xylofi/week-3/SalesForecasting_Superstore/summary.pdf). Below is a structured presentation of its core findings, metrics, and actionable recommendations.

### Executive Report Summary

#### 1. Executive Summary
We built an intelligent sales forecasting system using 4 years (2015-2018) of Superstore daily transaction data. Our best model (XGBoost) achieves a MAPE of 13.58% after correcting for data leakage, meaning forecasted sales are within 13.58% of actual values on average. We identified 3 distinct product demand clusters with tailored stocking strategies, and detected anomalous weeks driven by seasonal events and promotions.

#### 2. Key Findings from EDA & Forecasting
- **Revenue by Category:** Technology leads with $827K in total revenue, followed by Furniture ($729K) and Office Supplies ($705K).
- **Regional Performance:** West region shows the highest total sales and the most consistent year-over-year sales growth. East region shows the highest consistency in YoY growth rate trends (std dev of YoY growth is only 1.8%).
- **Shipping Efficiency:** Average shipping time is ~3.96 days across all regions. East region ships fastest (~3.91 days), while Central averages the longest at ~4.07 days.
- **Seasonality:** Strong Q4 spikes (October-December) driven by holiday demand. January-February consistently show the lowest sales.

#### 3. 3-Month Sales Forecast Model Comparison
We compared three different forecasting models on monthly sales data:

| Model | MAE | RMSE | MAPE (%) | Month 1 Forecast | Month 2 Forecast | Month 3 Forecast |
|---|---|---|---|---|---|---|
| **SARIMA** | $19,244.49 | $19,950.07 | 20.53% | $60,331.79 | $91,458.22 | $97,167.57 |
| **Prophet** | $20,296.01 | $22,487.47 | 21.89% | $51,083.66 | $90,045.40 | $89,661.19 |
| **XGBoost** | **$14,276.93** | **$19,568.64** | **13.58%** | **$86,465.82** | **$85,286.55** | **$84,191.90** |

*Recommendation:* XGBoost is recommended for production use — it has the lowest MAPE (13.58%) and the lowest MAE ($14,276.93). Forecasted sales for the next 3 months are in the range of $84K-$86K per month.

#### 4. Top 3 Anomalies Detected

| Date | Weekly Sales | Type | Likely Cause |
|---|---|---|---|
| **2015-03-22** | $37,703.67 | Extreme Spike | Possible corporate bulk order or promotional event (Q1 spike) |
| **2018-12-02** | $35,998.90 | Holiday Spike | Festive season / holiday sales rush (early December peak) |
| **2015-02-22** | $224.91 | Deep Slump | Post-holiday demand contraction (lowest recorded weekly sales) |

#### 5. Product Demand Segmentation & Stocking Strategy
- **High Volume, Stable Demand** (Phones, Chairs, Storage, Binders, Accessories, Tables, Machines): Maintain higher safety stock (~14 days), establish automated reorder points to prevent stockouts.
- **Growing Demand** (Copiers): Gradually build up inventory, negotiate volume discounts with suppliers to improve margins.
- **Low Volume, Volatile Demand** (Art, Fasteners, Paper, Labels, Envelopes, Bookcases, Furnishings, Supplies): Implement Just-In-Time (JIT) replenishment to minimize holding costs, and launch bundle promos to smooth out sales spikes.

#### 6. Actionable Business Recommendations
1. **Q4 Supply Chain Preparedness:** Because sales surge by over 200% in Q4 compared to Q1, supply chains must increase stocking levels by late September. Focus safety stock buffers specifically on the High Volume, Stable Demand clusters to capture peak holiday revenues without capital lockup.
2. **Capital Allocations for High-Growth Sectors:** Allocate marketing budgets and procurement pipelines to support the Growing Demand cluster (Copiers), which exhibits an exceptional year-over-year trajectory, while scaling down holding capacity for the Low Volume, Volatile group to optimize warehouse efficiency.